In [2]:
# ============================================================
# PACKAGE 11
# ENFORCEMENT INTELLIGENCE & PRIORITISATION AGENT
# ============================================================


import pandas as pd
import numpy as np
import json
import os


print("="*75)
print("PACKAGE 11 : ENFORCEMENT INTELLIGENCE")
print("="*75)



# ============================================================
# CONFIGURATION
# ============================================================


WARD_ATTRIBUTION = (
r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Geospatial_Pollution_Source_Attribution_Engine\Source_Attribution_Confidence_Calibration\AI_Ward_Source_Attribution.csv"
)


WARD_SOURCE = (
r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Geospatial_Pollution_Source_Attribution_Engine\Ward_Level_Source_Attribution_Outputs\Ward_Source_Attribution.csv"
)



OUTPUT_FOLDER = (
r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Enforcement_Intelligence_and_Prioritisation_Agent"
)


os.makedirs(
OUTPUT_FOLDER,
exist_ok=True
)



print("\nConfiguration Loaded")



# ============================================================
# LOAD DATA
# ============================================================


print("\nLoading Attribution Data...")


ai_df = pd.read_csv(
    WARD_ATTRIBUTION
)


ward_df = pd.read_csv(
    WARD_SOURCE
)



print(
"AI Attribution:",
ai_df.shape
)


print(
"Ward Attribution:",
ward_df.shape
)



# ============================================================
# MERGE
# ============================================================


df = ward_df.merge(

ai_df,

on="Ward_Name",

how="left"

)



print(
"\nMerged Dataset:",
df.shape
)

# ============================================================
# FIX DUPLICATE COLUMN NAMES AFTER MERGE
# ============================================================

print("\nColumns After Merge:")
print(df.columns.tolist())


# Use Ward Attribution as primary source attribution
if "Dominant_Source_x" in df.columns:

    df["Dominant_Source"] = df["Dominant_Source_x"]

elif "Dominant_Source_y" in df.columns:

    df["Dominant_Source"] = df["Dominant_Source_y"]


if "Confidence_x" in df.columns:

    df["Confidence"] = df["Confidence_x"]

elif "Confidence_y" in df.columns:

    df["Confidence"] = df["Confidence_y"]


# Remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]


print("\nFixed Columns:")
print(df.columns.tolist())

# ============================================================
# POLLUTION SEVERITY ESTIMATION
# ============================================================


print("\nCalculating Pollution Severity...")


# Using source intensity proxy
# because AQI prediction is city-level currently


df["Pollution_Severity"]=(

df["Traffic_%"]

+

df["Construction_%"]

+

df["Industrial_%"]

)/3



df["Pollution_Severity"]=(
df["Pollution_Severity"]
.clip(0,100)
)



# ============================================================
# SOURCE CONFIDENCE
# ============================================================


df["Source_Confidence"] = (

df["AI_Confidence_%"]

)



# ============================================================
# SENSITIVE LOCATION RISK
# ============================================================


df["Sensitive_Risk"] = (

df["Sensitive_Count"]

/


df["Sensitive_Count"]
.max()

*100

)


df["Sensitive_Risk"] = (

df["Sensitive_Risk"]
.fillna(0)

)



# ============================================================
# SOURCE DENSITY
# ============================================================


df["Source_Density"]=(

df["Traffic_Feature_Count"]

+

df["Construction_Count"]*5

+

df["Industrial_Count"]*10

)



df["Source_Density"]=(

df["Source_Density"]

/

df["Source_Density"].max()

*100

)



# ============================================================
# FINAL PRIORITY SCORE
# ============================================================


df["Enforcement_Priority_Score"]=(


0.40*
df["Pollution_Severity"]

+

0.30*
df["Source_Confidence"]

+

0.20*
df["Sensitive_Risk"]

+

0.10*
df["Source_Density"]


)



# ============================================================
# PRIORITY LEVEL
# ============================================================


def priority(x):

    if x>=75:
        return "Critical"

    elif x>=50:
        return "High"

    elif x>=30:
        return "Medium"

    else:
        return "Low"



df["Priority_Level"] = (

df["Enforcement_Priority_Score"]
.apply(priority)

)



# ============================================================
# ACTION RECOMMENDATION
# ============================================================



def action(source):


    if "Traffic" in source:

        return (
        "Vehicle emission inspection; "
        "traffic management; "
        "diesel vehicle checking"
        )


    elif "Construction" in source:

        return (
        "Construction dust inspection; "
        "anti-smog compliance check; "
        "material covering verification"
        )


    elif "Industrial" in source:

        return (
        "Industrial emission audit; "
        "stack monitoring; "
        "pollution control verification"
        )


    else:

        return (
        "General pollution monitoring"
        )



df["Recommended_Action"]=(

df["Dominant_Source"]

.apply(action)

)



# ============================================================
# SORT RANKING
# ============================================================



ranking = (

df.sort_values(

"Enforcement_Priority_Score",

ascending=False

)

.reset_index(drop=True)

)



ranking.insert(

0,

"Rank",

range(
1,
len(ranking)+1
)

)



# ============================================================
# SAVE CSV
# ============================================================


ranking.to_csv(

os.path.join(
OUTPUT_FOLDER,
"Enforcement_Priority_Ranking.csv"
),

index=False

)



ranking[

[
"Ward_Name",
"Ward_No",
"Dominant_Source",
"AI_Confidence_%",
"Enforcement_Priority_Score",
"Priority_Level",
"Recommended_Action"

]

].to_csv(

os.path.join(
OUTPUT_FOLDER,
"Ward_Enforcement_Action_Report.csv"
),

index=False

)



# ============================================================
# FRONTEND JSON
# ============================================================


frontend = ranking.to_dict(
orient="records"
)



with open(

os.path.join(
OUTPUT_FOLDER,
"Frontend_Enforcement_Intelligence.json"
),

"w"

) as f:


    json.dump(

    frontend,

    f,

    indent=4

    )



# ============================================================
# SUMMARY
# ============================================================


top = ranking.head(10)



with open(

os.path.join(
OUTPUT_FOLDER,
"Enforcement_Summary_Report.txt"
),

"w"

) as f:


    f.write(

"""
PACKAGE 11
ENFORCEMENT INTELLIGENCE REPORT


Top Priority Areas:

"""

)


    f.write(

top[
[
"Ward_Name",
"Dominant_Source",
"Priority_Level"
]

].to_string()

)



with open(

os.path.join(
OUTPUT_FOLDER,
"Package11_Metadata.txt"
),

"w"

) as f:


    f.write(

"""
PACKAGE 11

Purpose:
Generate evidence-backed pollution enforcement actions.

Scoring:

40% Pollution Severity
30% AI Source Confidence
20% Sensitive Population Risk
10% Source Density


Outputs:

Priority Ranking
Action Recommendation
Frontend JSON

"""

)



print("\n============================================================")
print("PACKAGE 11 COMPLETED SUCCESSFULLY")
print("============================================================")


print(
"""
Generated Files:

1. Enforcement_Priority_Ranking.csv

2. Ward_Enforcement_Action_Report.csv

3. Frontend_Enforcement_Intelligence.json

4. Enforcement_Summary_Report.txt

5. Package11_Metadata.txt

"""
)

PACKAGE 11 : ENFORCEMENT INTELLIGENCE

Configuration Loaded

Loading Attribution Data...
AI Attribution: (289, 4)
Ward Attribution: (290, 15)

Merged Dataset: (290, 18)

Columns After Merge:
['Ward_Name', 'Ward_No', 'Traffic_Feature_Count', 'Construction_Count', 'Industrial_Count', 'Sensitive_Count', 'Traffic_Score', 'Construction_Score', 'Industrial_Score', 'Sensitivity_Score', 'Traffic_%', 'Construction_%', 'Industrial_%', 'Dominant_Source_x', 'Confidence', 'Dominant_Source_y', 'AI_Confidence_%', 'Confidence_Level']

Fixed Columns:
['Ward_Name', 'Ward_No', 'Traffic_Feature_Count', 'Construction_Count', 'Industrial_Count', 'Sensitive_Count', 'Traffic_Score', 'Construction_Score', 'Industrial_Score', 'Sensitivity_Score', 'Traffic_%', 'Construction_%', 'Industrial_%', 'Dominant_Source_x', 'Confidence', 'Dominant_Source_y', 'AI_Confidence_%', 'Confidence_Level', 'Dominant_Source']

Calculating Pollution Severity...

PACKAGE 11 COMPLETED SUCCESSFULLY

Generated Files:

1. Enforcement_Prio